# Phase 3 - Model Architecture  

In this phase, the model architecture for the Facial Emotion Recognition system is designed and constructed. The objective is to develop an efficient deep learning model capable of extracting meaningful facial features while maintaining a balance between predictive performance and computational efficiency. The architecture development process includes building a lightweight baseline model for initial validation, followed by constructing the main FER model using a pretrained EfficientNet-B2 backbone combined with a custom classification head and attention mechanisms. A CBAM (Convolutional Block Attention Module) is integrated to enhance feature representation by allowing the network to focus on important spatial and channel information. Additional validation steps such as model summaries, parameter analysis, and forward-pass verification are also performed before proceeding to the training phase. EfficientNet-B2 is selected because it provides strong feature extraction performance with significantly fewer parameters than larger architectures such as ResNet-50, making it suitable for FER2013 images and reducing computational cost during training.

## 1. Imports, Device Setup, and Phase 2 Configuration

This section prepares the model architecture notebook by importing the required PyTorch modules and pretrained backbone architectures. It also detects the available hardware device and prints GPU information when CUDA is available. The configuration saved from Phase 2 is then loaded to restore important project settings such as the emotion classes, number of classes, and input image size. This ensures that the model architecture is built consistently with the data preparation pipeline.

In [1]:
import pickle, math
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import EfficientNet_B2_Weights, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load Phase 2 config
with open("eda_outputs/data_pipeline_config.pkl", "rb") as f:
    cfg = pickle.load(f)

EMOTIONS    = cfg["EMOTIONS"]
NUM_CLASSES = cfg["NUM_CLASSES"]
IMG_SIZE    = cfg["IMG_SIZE"]
print(f"\nClasses : {NUM_CLASSES}  ({', '.join(EMOTIONS)})")
print(f"Input   : {IMG_SIZE}x{IMG_SIZE}")


Device : cuda
GPU    : NVIDIA A100-SXM4-80GB
VRAM   : 85.0 GB

Classes : 7  (angry, disgust, fear, happy, neutral, sad, surprise)
Input   : 224x224


## 2. Model Utility Functions

This section defines utility functions used to inspect and validate model architectures. The parameter counting functions calculate the total, trainable, and frozen parameters, which helps compare model size and fine-tuning complexity. A forward-pass checking function is also included to test whether the model accepts the expected input shape and produces the correct output shape for the seven emotion classes. These utilities are useful for verifying each model architecture before moving to training, reducing the risk of hidden shape or configuration errors.

In [2]:
def count_params(model: nn.Module) -> dict:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    return {"total": total, "trainable": trainable, "frozen": frozen}


def print_param_summary(model: nn.Module, name: str = "Model"):
    p = count_params(model)
    print(f"── {name} parameter summary ───────────────────────────────")
    print(f"  Total      : {p['total']:>12,}")
    print(f"  Trainable  : {p['trainable']:>12,}")
    print(f"  Frozen     : {p['frozen']:>12,}")
    print(f"  Ratio      : {100*p['trainable']/p['total']:.1f}% trainable")


def forward_check(model: nn.Module, img_size: int = 224, batch: int = 4):
    """Run a dummy forward pass and print output shape."""
    model.eval()
    x = torch.randn(batch, 3, img_size, img_size).to(device)
    with torch.no_grad():
        out = model(x)
    print(f"  Input  : {tuple(x.shape)}")
    print(f"  Output : {tuple(out.shape)}  (expected: ({batch}, {NUM_CLASSES}))")
    assert out.shape == (batch, NUM_CLASSES), "Shape mismatch!"
    print("  ✓ Forward pass OK")
    return out


print("✓ Utilities defined")


✓ Utilities defined


## 3. CBAM Attention Module

This section implements the Convolutional Block Attention Module (CBAM), which enhances feature extraction by allowing the model to focus on the most informative regions and channels within feature maps.

CBAM consists of two sequential attention mechanisms:

- **Channel Attention:** learns which feature channels are more important by combining average and max pooling information across spatial dimensions.
- **Spatial Attention:** learns which spatial regions of the image contain the most relevant information by highlighting important locations in the feature maps.

Applying both mechanisms helps the model emphasize meaningful facial features and suppress irrelevant information, improving representation quality for facial emotion recognition.

A quick validation step is also performed to ensure that the CBAM module preserves the input shape and integrates correctly into the model architecture.

In [3]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        mid = max(in_channels // reduction, 8)   # floor at 8
        self.shared_mlp = nn.Sequential(
            nn.Linear(in_channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        # Average-pool and max-pool over spatial dims → (B, C)
        avg = x.mean(dim=[2, 3])
        mx  = x.amax(dim=[2, 3])
        # Shared MLP + combine
        gate = torch.sigmoid(self.shared_mlp(avg) + self.shared_mlp(mx))
        return x * gate.view(B, C, 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=pad, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compress channels: avg + max → (B, 2, H, W)
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.amax(dim=1, keepdim=True)
        cat = torch.cat([avg, mx], dim=1)
        gate = torch.sigmoid(self.conv(cat))
        return x * gate


class CBAM(nn.Module):
    """
    CBAM: Convolutional Block Attention Module
    Woo et al., ECCV 2018 — https://arxiv.org/abs/1807.06521
    """
    def __init__(self, in_channels: int, reduction: int = 16, spatial_k: int = 7):
        super().__init__()
        self.channel  = ChannelAttention(in_channels, reduction)
        self.spatial  = SpatialAttention(spatial_k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.channel(x)
        x = self.spatial(x)
        return x


# Quick test
_cbam = CBAM(64).to(device)
_x    = torch.randn(2, 64, 28, 28).to(device)
_y    = _cbam(_x)
assert _y.shape == _x.shape, "CBAM shape mismatch"
print(f"✓ CBAM OK — input {tuple(_x.shape)} → output {tuple(_y.shape)}")
print_param_summary(_cbam, "CBAM (64ch)")


✓ CBAM OK — input (2, 64, 28, 28) → output (2, 64, 28, 28)
── CBAM (64ch) parameter summary ───────────────────────────────
  Total      :        1,122
  Trainable  :        1,122
  Frozen     :            0
  Ratio      : 100.0% trainable


## 4. Custom FER Classification Head

This section implements a custom classification head for the Facial Emotion Recognition model. The classification head receives high-level features extracted by the pretrained backbone and transforms them into emotion class predictions. The architecture consists of multiple fully connected layers combined with batch normalization, activation functions, and dropout regularization. Batch normalization helps stabilize training, while dropout reduces overfitting by randomly disabling neurons during learning.

The implemented architecture follows the structure below:

**GlobalAvgPool → Flatten**  
**→ FC(backbone_dim → 512)**  
**→ BatchNorm → ReLU → Dropout(0.4)**  
**→ FC(512 → 256)**  
**→ BatchNorm → ReLU → Dropout(0.3)**  
**→ FC(256 → 7)**

The final output layer generates prediction scores for the seven emotion classes, allowing the model to classify each facial image into its corresponding emotion category.

In [4]:
class FERHead(nn.Module):
    """
    Custom classification head for FER.

    Args:
        in_features : output dim of backbone (1408 for EfficientNet-B2)
        num_classes : number of emotion classes (7)
        dropout1    : dropout after first FC layer
        dropout2    : dropout after second FC layer
    """
    def __init__(
        self,
        in_features : int,
        num_classes : int = 7,
        dropout1    : float = 0.4,
        dropout2    : float = 0.3,
    ):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout1),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout2),

            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(x)


print("✓ FERHead defined")


✓ FERHead defined


## 5. FERModel Architecture

This section implements the complete Facial Emotion Recognition model by combining a pretrained EfficientNet-B2 backbone, CBAM attention mechanism, and a custom classification head.

The model uses EfficientNet-B2 as the feature extractor because it provides strong performance with fewer parameters compared to larger architectures. Early backbone layers are partially frozen so that low-level visual features learned from ImageNet are preserved while reducing computational cost during training.

The overall architecture follows the structure below:

**Input Image (224×224×3)**  
**→ EfficientNet-B2 Backbone**  
**→ CBAM Attention Module**  
**→ Global Average Pooling**  
**→ Flatten**  
**→ FC(1408 → 512)**  
**→ BatchNorm → ReLU → Dropout(0.4)**  
**→ FC(512 → 256)**  
**→ BatchNorm → ReLU → Dropout(0.3)**  
**→ FC(256 → 7)**  
**→ Emotion Prediction**

Additional initialization is applied to newly added layers, and layer-wise learning rate groups are defined to support fine-tuning in the training phase. Lower learning rates are assigned to pretrained backbone layers, while the CBAM module and classification head receive larger learning rates for faster adaptation.

In [5]:
class FERModel(nn.Module):
    """
    Facial Emotion Recognition model.

    Architecture:
        EfficientNet-B2 backbone (pretrained ImageNet)
        → CBAM attention on final feature map
        → Global average pool
        → Custom FERHead (FC-BN-ReLU-Dropout × 2 → 7-way classifier)

    Args:
        num_classes     : number of emotion classes
        pretrained      : use ImageNet pretrained weights
        freeze_blocks   : freeze first N backbone blocks (0–8 for EffNet-B2)
        cbam_reduction  : channel reduction ratio for CBAM
        dropout1/2      : dropout rates in FERHead
    """

    BACKBONE_OUT_FEATURES = 1408   # EfficientNet-B2 feature dim

    def __init__(
        self,
        num_classes   : int   = 7,
        pretrained    : bool  = True,
        freeze_blocks : int   = 5,
        cbam_reduction: int   = 16,
        dropout1      : float = 0.4,
        dropout2      : float = 0.3,
    ):
        super().__init__()

        # ── Backbone ───────────────────────────────────────────────────────
        weights  = EfficientNet_B2_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.efficientnet_b2(weights=weights)

        # Remove the original classifier
        self.features = backbone.features   # Conv stem + 8 MBConv blocks
        self.avgpool  = backbone.avgpool    # AdaptiveAvgPool2d(1)

        # ── Freeze early blocks ────────────────────────────────────────────
        # features[0]      = stem Conv
        # features[1..8]   = MBConv blocks 1..8
        for i, block in enumerate(self.features):
            if i < freeze_blocks:
                for param in block.parameters():
                    param.requires_grad = False

        # ── CBAM on final feature map (1408 channels) ──────────────────────
        self.cbam = CBAM(self.BACKBONE_OUT_FEATURES, reduction=cbam_reduction)

        # ── Custom head ────────────────────────────────────────────────────
        self.classifier = FERHead(
            in_features=self.BACKBONE_OUT_FEATURES,
            num_classes=num_classes,
            dropout1=dropout1,
            dropout2=dropout2,
        )

        # ── Weight init for new layers ─────────────────────────────────────
        self._init_new_layers()

    def _init_new_layers(self):
        for m in list(self.cbam.modules()) + list(self.classifier.modules()):
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)       # (B, 1408, 7, 7)  for 224x224 input
        x = self.cbam(x)           # (B, 1408, 7, 7)  attention gating
        x = self.avgpool(x)        # (B, 1408, 1, 1)
        x = torch.flatten(x, 1)   # (B, 1408)
        x = self.classifier(x)    # (B, 7)
        return x

    def get_param_groups(self, base_lr: float) -> list[dict]:
        """
        Returns parameter groups with layer-wise LR decay for the optimizer.
        Phase 4 passes this directly to torch.optim.AdamW.
        """
        return [
            # Unfrozen backbone blocks (late layers) — small LR
            {
                "params": [p for p in self.features.parameters() if p.requires_grad],
                "lr": base_lr * 0.1,
                "name": "backbone_unfrozen",
            },
            # CBAM — full LR
            {
                "params": self.cbam.parameters(),
                "lr": base_lr,
                "name": "cbam",
            },
            # Head — full LR
            {
                "params": self.classifier.parameters(),
                "lr": base_lr,
                "name": "head",
            },
        ]


print("✓ FERModel defined")


✓ FERModel defined


## 6. Model Initialization and Parameter Analysis

This section initializes the complete FER model using the predefined architecture and configuration settings. The model is created using a pretrained EfficientNet-B2 backbone, integrated CBAM attention module, and the custom classification head developed earlier. After initialization, a detailed parameter analysis is performed to calculate the total number of parameters and distinguish between trainable and frozen layers. Since the early backbone blocks are frozen, only selected layers participate in weight updates during training.

The parameter breakdown is also displayed separately for the main components:

- **Backbone features** — pretrained feature extraction layers  
- **CBAM module** — attention mechanism layers  
- **Classification head** — fully connected prediction layers  

This analysis provides a clearer understanding of model complexity and confirms that the fine-tuning strategy is configured correctly before training begins.

In [6]:
model = FERModel(
    num_classes    = NUM_CLASSES,
    pretrained     = True,
    freeze_blocks  = 5,      # freeze stem + first 4 MBConv blocks
    cbam_reduction = 16,
    dropout1       = 0.4,
    dropout2       = 0.3,
).to(device)

print_param_summary(model, "FERModel (EfficientNet-B2 + CBAM)")
print()

# Per-group breakdown
print("── Parameter groups ─────────────────────────────────────────")
for name, module in [("features (backbone)", model.features),
                      ("cbam",               model.cbam),
                      ("classifier (head)",  model.classifier)]:
    total   = sum(p.numel() for p in module.parameters())
    trained = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"  {name:<26} total={total:>9,}  trainable={trained:>9,}")


── FERModel (EfficientNet-B2 + CBAM) parameter summary ───────────────────────────────
  Total      :    8,804,971
  Trainable  :    8,249,875
  Frozen     :      555,096
  Ratio      : 93.7% trainable

── Parameter groups ─────────────────────────────────────────
  features (backbone)        total=7,700,994  trainable=7,145,898
  cbam                       total=  247,906  trainable=  247,906
  classifier (head)          total=  856,071  trainable=  856,071


## 7. Forward Pass Validation and Feature Inspection

This section validates the complete model architecture by performing a dummy forward pass using synthetic input data. The goal is to confirm that the model accepts the expected input dimensions and produces output logits with the correct number of emotion classes. Intermediate feature shapes are also inspected throughout the network pipeline to verify that each component processes the data correctly. The following stages are examined:

- **Backbone output** — extracted feature maps from EfficientNet-B2  
- **CBAM output** — attention-enhanced feature maps  
- **Global average pooling output** — spatial compression of features  
- **Flatten output** — conversion into feature vectors  
- **Final logits** — prediction scores for the seven emotion classes  

Finally, the probability distribution from the untrained model is inspected. Since the model has not yet been trained, prediction probabilities are expected to be approximately uniform across all classes, indicating that the network initialization behaves correctly before training begins.

In [7]:
print("── Dummy forward pass ───────────────────────────────────────")
out = forward_check(model, img_size=IMG_SIZE, batch=4)

print()
print("── Intermediate feature shapes ──────────────────────────────")
model.eval()
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    feat  = model.features(x)
    print(f"  After backbone  : {tuple(feat.shape)}")
    att   = model.cbam(feat)
    print(f"  After CBAM      : {tuple(att.shape)}")
    pool  = model.avgpool(att)
    print(f"  After avgpool   : {tuple(pool.shape)}")
    flat  = torch.flatten(pool, 1)
    print(f"  After flatten   : {tuple(flat.shape)}")
    logit = model.classifier(flat)
    print(f"  Output logits   : {tuple(logit.shape)}")

print()
print("── Logit sanity (untrained — should be near uniform) ────────")
probs = F.softmax(out, dim=-1)
print(f"  Mean prob per class : {probs.mean(0).detach().cpu().numpy().round(3)}")
print(f"  (Ideal untrained    : ~{1/NUM_CLASSES:.3f} each)")


── Dummy forward pass ───────────────────────────────────────
  Input  : (4, 3, 224, 224)
  Output : (4, 7)  (expected: (4, 7))
  ✓ Forward pass OK

── Intermediate feature shapes ──────────────────────────────
  After backbone  : (2, 1408, 7, 7)
  After CBAM      : (2, 1408, 7, 7)
  After avgpool   : (2, 1408, 1, 1)
  After flatten   : (2, 1408)
  Output logits   : (2, 7)

── Logit sanity (untrained — should be near uniform) ────────
  Mean prob per class : [0.367 0.017 0.021 0.163 0.067 0.255 0.109]
  (Ideal untrained    : ~0.143 each)


## 8. Baseline CNN for Sanity Check

This section implements a lightweight baseline CNN model that serves as an initial reference architecture for validating the training pipeline. The model consists of four convolutional blocks followed by a simple classification head. The architecture gradually extracts hierarchical visual features through convolution, batch normalization, activation functions, and max-pooling operations. After feature extraction, global average pooling and a fully connected layer are used to generate predictions for the seven emotion classes. The baseline model is intentionally designed with a small number of parameters (~390K) to provide fast training and debugging capabilities before training the more complex EfficientNet-B2 + CBAM architecture. A forward-pass validation is also performed to ensure that the model produces outputs with the expected dimensions and integrates correctly into the overall pipeline.

In [ ]:
class BaselineCNN(nn.Module):
    """
    Lightweight 4-block CNN for pipeline sanity-checking.
    ~390k parameters. Trains in ~5 min on A100.
    """
    def __init__(self, num_classes: int = 7):
        super().__init__()
        def conv_block(in_c, out_c, pool=True):
            layers = [
                nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
            ]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.encoder = nn.Sequential(
            conv_block(3,   32),    # 224 → 112
            conv_block(32,  64),    # 112 → 56
            conv_block(64,  128),   # 56  → 28
            conv_block(128, 256),   # 28  → 14
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.encoder(x))


baseline = BaselineCNN(NUM_CLASSES).to(device)
print_param_summary(baseline, "BaselineCNN")
print()
print("── Forward pass ──────────────────────────────────────────────")
forward_check(baseline, img_size=IMG_SIZE, batch=4)


── BaselineCNN parameter summary ───────────────────────────────
  Total      :      390,695
  Trainable  :      390,695
  Frozen     :            0
  Ratio      : 100.0% trainable

── Forward pass ──────────────────────────────────────────────
  Input  : (4, 3, 224, 224)
  Output : (4, 7)  (expected: (4, 7))
  ✓ Forward pass OK


tensor([[ 0.0245, -0.0447,  0.0371,  0.0318,  0.1773, -0.0214, -0.0407],
        [ 0.0244, -0.0455,  0.0361,  0.0327,  0.1774, -0.0223, -0.0395],
        [ 0.0237, -0.0439,  0.0369,  0.0332,  0.1776, -0.0221, -0.0397],
        [ 0.0245, -0.0456,  0.0368,  0.0323,  0.1782, -0.0219, -0.0397]],
       device='cuda:0')

## 9. Model Throughput Benchmark

This section benchmarks the inference speed of both the lightweight baseline CNN and the main FERModel architecture. The benchmark measures how long each model takes to process a batch of images and reports the approximate number of images processed per second. A short warmup phase is performed before timing to ensure more stable GPU measurements. The comparison helps evaluate the trade-off between model complexity and computational efficiency. This analysis confirms whether the selected EfficientNet-B2 + CBAM model remains practical for training and inference while offering stronger representational power than the simple baseline CNN.

In [9]:
import time

def benchmark(model, img_size, batch=64, n_iters=20, label=""):
    model.eval().to(device)
    x = torch.randn(batch, 3, img_size, img_size).to(device)
    # Warmup
    with torch.no_grad():
        for _ in range(3):
            model(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_iters):
            model(x)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / n_iters * 1000
    p = count_params(model)
    print(f"  {label:<30} params={p['trainable']:>9,}  "
          f"batch_ms={elapsed:6.1f}  "
          f"imgs/s={batch/(elapsed/1000):>7.0f}")

print("── Throughput benchmark (batch=64, A100) ────────────────────")
benchmark(baseline, IMG_SIZE,  label="BaselineCNN")
benchmark(model,    IMG_SIZE,  label="FERModel (EffNet-B2+CBAM)")


── Throughput benchmark (batch=64, A100) ────────────────────


  BaselineCNN                    params=  390,695  batch_ms=   5.2  imgs/s=  12263
  FERModel (EffNet-B2+CBAM)      params=8,249,875  batch_ms=  19.0  imgs/s=   3371


## 10 — Saving Model Architecture Checkpoint

This final section saves the initialized model architecture and configuration settings as a checkpoint file for future phases. The checkpoint stores the model parameters, architecture configuration, and Phase 2 preprocessing settings required for reproducibility. Saving the initial checkpoint allows the training phase to reload the same architecture without rebuilding all components manually. It also ensures that model settings such as backbone configuration, attention parameters, and dropout values remain consistent across experiments. The saved checkpoint acts as a reusable starting point for Phase 4, where the training strategy and optimization process will be applied.

In [10]:
import os
os.makedirs("checkpoints", exist_ok=True)

# Save architecture config + initial state dict
checkpoint = {
    "model_state"  : model.state_dict(),
    "model_config" : {
        "num_classes"    : NUM_CLASSES,
        "pretrained"     : True,
        "freeze_blocks"  : 5,
        "cbam_reduction" : 16,
        "dropout1"       : 0.4,
        "dropout2"       : 0.3,
    },
    "phase2_config" : cfg,
    "architecture"  : "EfficientNet-B2 + CBAM + FERHead",
}

ckpt_path = "checkpoints/phase3_init.pt"
torch.save(checkpoint, ckpt_path)
print(f"✓ Saved initial checkpoint → {ckpt_path}")
print(f"  Size: {os.path.getsize(ckpt_path)/1e6:.1f} MB")


✓ Saved initial checkpoint → checkpoints/phase3_init.pt
  Size: 35.7 MB
